# ColonyNet: Two-Stage Training (One Notebook)

This notebook runs a full 2-stage training pipeline on `trainable_pool`:

1. **Stage 1 (pretrain):** all samples **except** `data_cups*`
2. **Stage 2 (finetune):** only `data_cups*`, initialized from Stage 1 `best.pt`

It uses:
- `tools/make_two_stage_ids.py`
- `configs/train_trainable_pool_stage1_pretrain_no_cups_mit_b3.yaml`
- `configs/train_trainable_pool_stage2_finetune_cups_mit_b3.yaml`


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import yaml

PROJECT_ROOT = Path('.').resolve()
VENV_PYTHON = PROJECT_ROOT / '.venv' / 'Scripts' / 'python.exe'
if not VENV_PYTHON.exists():
    raise FileNotFoundError(f'Python from .venv not found: {VENV_PYTHON}')

print('Project root:', PROJECT_ROOT)
print('Python:', VENV_PYTHON)


In [ ]:
# Paths and configs
SPLITS_DIR = Path('data/splits')
CFG_STAGE1 = Path('configs/train_trainable_pool_stage1_pretrain_no_cups_mit_b3.yaml')
CFG_STAGE2 = Path('configs/train_trainable_pool_stage2_finetune_cups_mit_b3.yaml')

LOG_STAGE1 = Path('runs/stage1_train.log')
LOG_STAGE2 = Path('runs/stage2_train.log')

assert CFG_STAGE1.exists(), f'Missing config: {CFG_STAGE1}'
assert CFG_STAGE2.exists(), f'Missing config: {CFG_STAGE2}'

print('Stage1 config:', CFG_STAGE1)
print('Stage2 config:', CFG_STAGE2)
print('Stage1 log:', LOG_STAGE1)
print('Stage2 log:', LOG_STAGE2)


In [ ]:
# Helper to run shell commands with notebook-friendly progress + realtime curves
import re
from tqdm.auto import tqdm
from IPython.display import display
import matplotlib.pyplot as plt


def run_cmd(
    cmd,
    cwd='.',
    log_path=None,
    epoch_total=None,
    quiet_tqdm_lines=True,
    live_plots=True,
    live_plot_every=1,
):
    print('\n>>>', ' '.join(cmd))

    log_f = None
    if log_path is not None:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_f = log_path.open('w', encoding='utf-8')
        print('logging to:', log_path)

    env = os.environ.copy()
    env['PYTHONIOENCODING'] = 'utf-8'

    train_pbar = None
    val_pbar = None
    current_epoch = None

    hist_train_loss = []
    hist_val_f1 = []
    plot_handle = None

    train_re = re.compile(r"Epoch\s+(\d+)/(\d+)\s+\[train\]:.*?\|\s*(\d+)/(\d+)\s*\[.*loss=([0-9.]+)")
    val_re = re.compile(r"Epoch\s+(\d+)/(\d+)\s+\[val\]:.*?\|\s*(\d+)/(\d+)\s*\[")
    epoch_summary_re = re.compile(
        r"Epoch\s+(\d+)/(\d+)\s*\|\s*train_loss=([0-9.]+)\s*\|\s*val_f1=([0-9.]+)\s*\|\s*merge=([0-9.]+)\s*\|\s*split=([0-9.]+)\s*\|\s*count_err=([0-9.]+)"
    )

    def redraw_curves(final=False):
        nonlocal plot_handle
        if not live_plots or not hist_train_loss:
            return
        if (not final) and (len(hist_train_loss) % max(1, int(live_plot_every)) != 0):
            return

        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        ax[0].plot(hist_train_loss, label='train_loss')
        ax[0].set_title('Train Loss')
        ax[0].legend()

        ax[1].plot(hist_val_f1, label='val_f1')
        ax[1].set_title('Val F1')
        ax[1].legend()

        if plot_handle is None:
            plot_handle = display(fig, display_id=True)
        else:
            plot_handle.update(fig)
        plt.close(fig)

    def ensure_epoch(epoch_num):
        nonlocal current_epoch, train_pbar, val_pbar
        if current_epoch == epoch_num:
            return

        if train_pbar is not None:
            train_pbar.close()
            train_pbar = None
        if val_pbar is not None:
            val_pbar.close()
            val_pbar = None

        current_epoch = epoch_num

    try:
        proc = subprocess.Popen(
            cmd,
            cwd=str(Path(cwd).resolve()),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
            env=env,
        )

        for raw in proc.stdout:
            if log_f is not None:
                log_f.write(raw)

            line = raw.rstrip('\n')

            mt = train_re.search(line)
            if mt:
                ep = int(mt.group(1)); ep_tot = int(mt.group(2))
                cur = int(mt.group(3)); tot = int(mt.group(4)); loss = float(mt.group(5))
                ensure_epoch(ep)
                if train_pbar is None or train_pbar.total != tot:
                    if train_pbar is not None:
                        train_pbar.close()
                    train_pbar = tqdm(total=tot, desc=f'Epoch {ep}/{ep_tot} [train]')
                if cur >= train_pbar.n:
                    train_pbar.update(cur - train_pbar.n)
                train_pbar.set_postfix(loss=f'{loss:.4f}')
                continue

            mv = val_re.search(line)
            if mv:
                ep = int(mv.group(1)); ep_tot = int(mv.group(2))
                cur = int(mv.group(3)); tot = int(mv.group(4))
                ensure_epoch(ep)
                if val_pbar is None or val_pbar.total != tot:
                    if val_pbar is not None:
                        val_pbar.close()
                    val_pbar = tqdm(total=tot, desc=f'Epoch {ep}/{ep_tot} [val]')
                if cur >= val_pbar.n:
                    val_pbar.update(cur - val_pbar.n)
                continue

            me = epoch_summary_re.search(line)
            if me:
                tl = float(me.group(3)); vf1 = float(me.group(4))
                hist_train_loss.append(tl)
                hist_val_f1.append(vf1)
                redraw_curves(final=False)
                print(line)
                continue

            if line.startswith('Epoch time:'):
                print(line)
                continue

            if 'Saved best:' in line or 'Loaded init checkpoint:' in line:
                print(line)
            elif not quiet_tqdm_lines and line:
                print(line)

        proc.wait()
    finally:
        redraw_curves(final=True)
        if train_pbar is not None:
            train_pbar.close()
        if val_pbar is not None:
            val_pbar.close()
        if log_f is not None:
            log_f.close()

    if proc.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {proc.returncode}: {cmd}')




In [ ]:
# Step 1: generate/re-generate two-stage ID splits
run_cmd([
    str(VENV_PYTHON),
    'tools/make_two_stage_ids.py',
    '--images_dir', 'trainable_pool/images',
    '--instances_dir', 'trainable_pool/instances',
    '--out_dir', str(SPLITS_DIR),
    '--cups_prefix', 'data_cups',
    '--val_split_stage1', '0.15',
    '--val_split_stage2', '0.15',
    '--seed', '42',
])


In [ ]:
# Show split sizes
for p in [
    SPLITS_DIR / 'stage1_pretrain_train_ids.txt',
    SPLITS_DIR / 'stage1_pretrain_val_ids.txt',
    SPLITS_DIR / 'stage2_finetune_train_ids.txt',
    SPLITS_DIR / 'stage2_finetune_val_ids.txt',
]:
    n = len([x for x in p.read_text(encoding='utf-8').splitlines() if x.strip()])
    print(f'{p}: {n}')


In [ ]:
# Step 2: Stage 1 training (pretrain on non-cups)
cfg1_tmp = yaml.safe_load(CFG_STAGE1.read_text(encoding='utf-8'))
run_cmd([
    str(VENV_PYTHON),
    'train.py',
    '--config', str(CFG_STAGE1),
    '--runs_dir', 'runs',
], log_path=LOG_STAGE1, epoch_total=int(cfg1_tmp['train']['epochs']), quiet_tqdm_lines=True)


In [ ]:
# Verify Stage 1 checkpoint exists
cfg1 = yaml.safe_load(CFG_STAGE1.read_text(encoding='utf-8'))
stage1_run = Path('runs') / cfg1['run_name']
stage1_best = stage1_run / 'best.pt'
stage1_last = stage1_run / 'last.pt'

print('Stage1 run dir:', stage1_run)
print('best exists:', stage1_best.exists(), stage1_best)
print('last exists:', stage1_last.exists(), stage1_last)

if not stage1_best.exists():
    raise FileNotFoundError(f'Stage1 best checkpoint not found: {stage1_best}')


In [ ]:
# Step 3: Stage 2 training (finetune on cups only)
# Config already points to runs/colony_stage1_pretrain_no_cups_mit_b3/best.pt as init_ckpt.
cfg2_tmp = yaml.safe_load(CFG_STAGE2.read_text(encoding='utf-8'))
run_cmd([
    str(VENV_PYTHON),
    'train.py',
    '--config', str(CFG_STAGE2),
    '--runs_dir', 'runs',
], log_path=LOG_STAGE2, epoch_total=int(cfg2_tmp['train']['epochs']), quiet_tqdm_lines=True)


In [ ]:
# Verify Stage 2 checkpoints
cfg2 = yaml.safe_load(CFG_STAGE2.read_text(encoding='utf-8'))
stage2_run = Path('runs') / cfg2['run_name']
stage2_best = stage2_run / 'best.pt'
stage2_last = stage2_run / 'last.pt'

print('Stage2 run dir:', stage2_run)
print('best exists:', stage2_best.exists(), stage2_best)
print('last exists:', stage2_last.exists(), stage2_last)


In [ ]:
# Training curves from logs
import re
import numpy as np
import matplotlib.pyplot as plt


def parse_train_log(path):
    path = Path(path)
    if not path.exists():
        print(f'log missing: {path}')
        return None

    epoch, train_loss = [], []
    val_f1, val_merge, val_split, val_count = [], [], [], []

    pat = re.compile(
        r"Epoch\s+(\d+)/(\d+)\s*\|\s*train_loss=([0-9.]+)\s*\|\s*val_f1=([0-9.]+)\s*\|\s*merge=([0-9.]+)\s*\|\s*split=([0-9.]+)\s*\|\s*count_err=([0-9.]+)"
    )

    for ln in path.read_text(encoding='utf-8', errors='ignore').splitlines():
        m = pat.search(ln)
        if not m:
            continue
        epoch.append(int(m.group(1)))
        train_loss.append(float(m.group(3)))
        val_f1.append(float(m.group(4)))
        val_merge.append(float(m.group(5)))
        val_split.append(float(m.group(6)))
        val_count.append(float(m.group(7)))

    if not epoch:
        print(f'no parsed epoch lines in: {path}')
        return None

    return {
        'epoch': np.array(epoch),
        'train_loss': np.array(train_loss),
        'val_f1': np.array(val_f1),
        'val_merge': np.array(val_merge),
        'val_split': np.array(val_split),
        'val_count_err': np.array(val_count),
        'path': path,
    }


def plot_stage_curves(data, title):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))

    ax[0].plot(data['train_loss'], label='train_loss')
    ax[0].set_title('Train Loss')
    ax[0].legend()

    ax[1].plot(data['val_f1'], label='val_f1')
    ax[1].set_title('Val F1')
    ax[1].legend()

    print(title)
    plt.show()


stage1 = parse_train_log(LOG_STAGE1)
stage2 = parse_train_log(LOG_STAGE2)

if stage1 is not None:
    print('Stage1 epochs:', len(stage1['epoch']), 'best val_f1:', float(stage1['val_f1'].max()))
    plot_stage_curves(stage1, 'Stage 1: Pretrain (non-cups)')

if stage2 is not None:
    print('Stage2 epochs:', len(stage2['epoch']), 'best val_f1:', float(stage2['val_f1'].max()))
    plot_stage_curves(stage2, 'Stage 2: Finetune (cups)')



## Notes

- If GPU memory is tight, reduce `batch_size` in both stage configs.
- If Stage 2 overfits quickly, reduce `epochs` or increase `val_split_stage2`.
- You can restart only Stage 2 after updating postprocess/loss settings.


In [ ]:
# Final visualization cell (similar to train_unified_b3.ipynb)
from pathlib import Path
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.segmentation import find_boundaries

from colonyseg.models.colonymet import ColonyNet
from colonyseg.post.watershed import postprocess_watershed

# === paths ===
default_ckpt = Path('runs/colony_stage2_finetune_cups_mit_b3/best.pt')
ckpt_path = str(stage2_best) if 'stage2_best' in globals() and Path(stage2_best).exists() else str(default_ckpt)
img_path = 'IMG_4377.jpg'

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# === load model ===
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
cfg = ckpt['cfg']
model = ColonyNet(backbone_id=cfg['model']['backbone_id'],
                  fpn_dim=int(cfg['model']['fpn_dim'])).to(device)
model.load_state_dict(ckpt['model'], strict=True)
model.eval()

# === petri crop helpers ===
def detect_petri_circle(img_rgb, min_r_frac=0.35, max_r_frac=0.55, center_tol=0.25):
    h, w = img_rgb.shape[:2]
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (9, 9), 2)

    min_r = int(min(h, w) * min_r_frac)
    max_r = int(min(h, w) * max_r_frac)

    circles = cv2.HoughCircles(
        gray, cv2.HOUGH_GRADIENT, dp=1.2, minDist=min(h, w)//2,
        param1=100, param2=30, minRadius=min_r, maxRadius=max_r
    )
    if circles is not None:
        circles = np.round(circles[0]).astype(int)
        cx, cy, r = circles[np.argmax(circles[:,2])]
    else:
        edges = cv2.Canny(gray, 50, 150)
        cnts, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not cnts:
            return None
        cnt = max(cnts, key=cv2.contourArea)
        (cx_f, cy_f), r_f = cv2.minEnclosingCircle(cnt)
        cx, cy, r = int(cx_f), int(cy_f), int(r_f)

    if r < min_r or r > max_r:
        return None
    cx0, cy0 = w // 2, h // 2
    max_off = center_tol * min(h, w)
    if ((cx - cx0)**2 + (cy - cy0)**2) ** 0.5 > max_off:
        return None
    return cx, cy, r


def crop_petri(img_rgb, pad=0.02, mask_outside=True):
    h, w = img_rgb.shape[:2]
    circ = detect_petri_circle(img_rgb)
    if circ is None:
        return img_rgb, None, 'no_circle'
    cx, cy, r = circ
    r = int(r * (1.0 + pad))

    x1, y1 = max(0, cx - r), max(0, cy - r)
    x2, y2 = min(w, cx + r), min(h, cy + r)

    crop = img_rgb[y1:y2, x1:x2].copy()

    if mask_outside:
        yy, xx = np.ogrid[y1:y2, x1:x2]
        mask = (xx - cx) ** 2 + (yy - cy) ** 2 <= (r * r)
        crop[~mask] = 0

    return crop, (cx, cy, r, x1, y1, x2, y2), 'cropped'


def overlay_boundaries(img, lbl):
    out = img.copy()
    b = find_boundaries(lbl, mode='outer')
    out[b] = (0, 255, 0)
    return out


# === load image + petri crop ===
img_bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
if img_bgr is None:
    raise FileNotFoundError(f'Image not found: {img_path}')

img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_petri, meta, status = crop_petri(img_rgb, pad=0.02, mask_outside=True)

# === resize to model input ===
img_rs = cv2.resize(img_petri, (cfg['data']['img_size'], cfg['data']['img_size']),
                    interpolation=cv2.INTER_AREA)

# === inference ===
x = torch.from_numpy(img_rs).float().permute(2,0,1) / 255.0
x = x.unsqueeze(0).to(device)

with torch.no_grad():
    pred = model(x)
    sem_p = torch.sigmoid(pred['sem']).cpu().numpy()[0,0]
    cen_p = torch.sigmoid(pred['center']).cpu().numpy()[0,0]
    bnd_p = torch.sigmoid(pred['boundary']).cpu().numpy()[0,0]

# Upsample head outputs to input resolution before watershed for smoother boundaries
h_in, w_in = img_rs.shape[:2]
h_out, w_out = sem_p.shape
scale = h_in / float(h_out)

sem_hi = cv2.resize(sem_p.astype(np.float32), (w_in, h_in), interpolation=cv2.INTER_LINEAR)
cen_hi = cv2.resize(cen_p.astype(np.float32), (w_in, h_in), interpolation=cv2.INTER_LINEAR)
bnd_hi = cv2.resize(bnd_p.astype(np.float32), (w_in, h_in), interpolation=cv2.INTER_LINEAR)

min_distance_hi = max(1, int(round(float(cfg['post']['min_distance']) * scale)))
area_min_hi = max(1, int(round(float(cfg['post']['area_min']) * (scale ** 2))))
area_max_hi = int(round(float(cfg['post']['area_max']) * (scale ** 2)))

labels_up = postprocess_watershed(
    sem_hi, cen_hi, bnd_hi,
    t_sem=float(cfg['post']['t_sem']),
    t_center=float(cfg['post']['t_center']),
    min_distance=min_distance_hi,
    lambda_boundary=float(cfg['post']['lambda_boundary']),
    area_min=area_min_hi,
    area_max=area_max_hi,
)

# === visualize ===
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].set_title('Original')
ax[0].imshow(img_rgb); ax[0].axis('off')

ax[1].set_title(f'Petri-cropped ({status})')
ax[1].imshow(img_petri); ax[1].axis('off')

ax[2].set_title('Pred boundaries')
ax[2].imshow(overlay_boundaries(img_rs, labels_up)); ax[2].axis('off')
plt.show()


In [ ]:
import torch
print(torch.__version__)